### Initialize Cohere Reranker

Configure Cohere Rerank 5 Fast as the hosted Cross-Encoder replacement.

The API will perform query-specific ranking of scheme profiles, eliminating local CPU inference.

In [2]:
import os
import cohere
from dotenv import load_dotenv

load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

if not COHERE_API_KEY:
    raise ValueError("COHERE_API_KEY not found in environment.")

co = cohere.ClientV2(api_key=COHERE_API_KEY)

RERANKER_MODEL = "rerank-v5.0-fast"

print("Cohere client initialized successfully.")
print("Model:", RERANKER_MODEL)

Cohere client initialized successfully.
Model: rerank-v5.0-fast


### Cohere Reranker – Single Query Test

Evaluate Cohere Rerank 5 Fast on one query from the golden dataset.

Use the existing Hybrid RRF candidate schemes and their representative profiles. Measure API latency and inspect the top-ranked scheme IDs and relevance scores.

In [4]:
import json
import os
from pathlib import Path
import pandas as pd

h_result = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\hybrid_rrf_results.jsonl")

with open (h_result, "r") as f:
    hybrid_df = pd.read_json(f, lines=True)

hybrid_df.head(5)

,test_id,relevant_scheme_ids,ranked_scheme_ids,rrf_results
0,DISC-0001,[3he_F5gB5uF80_MGg5Hz],"[3he_F5gB5uF80_MGg5Hz, GMZdU5kB0AGc0F0ifzmY, h...","[{'scheme_id': '3he_F5gB5uF80_MGg5Hz', 'rrf_sc..."
1,DISC-0002,[3he_F5gB5uF80_MGg5Hz],"[hca_2ZoB0AGc0F0iCPyl, yKKDP54BXu-iUKXqClWA, B...","[{'scheme_id': 'hca_2ZoB0AGc0F0iCPyl', 'rrf_sc..."
2,DISC-0003,[3he_F5gB5uF80_MGg5Hz],"[caJdDZwBXu-iUKXq_Ubo, GBhS0ZwB5uF80_MGu2Cf, I...","[{'scheme_id': 'caJdDZwBXu-iUKXq_Ubo', 'rrf_sc..."
3,DISC-0004,[3he_F5gB5uF80_MGg5Hz],"[7BcqTpkB5uF80_MGypQF, 6cZHxJoB0AGc0F0iffvh, Y...","[{'scheme_id': '7BcqTpkB5uF80_MGypQF', 'rrf_sc..."
4,DISC-0005,[3he_F5gB5uF80_MGg5Hz],"[BRjnxJoB5uF80_MGyVbp, GMZscpoB0AGc0F0iyj3X, h...","[{'scheme_id': 'BRjnxJoB5uF80_MGyVbp', 'rrf_sc..."


### Load Existing Discovery Vector Database

Connect to the previously persisted seva_discovery_v1 Chroma database.

We will reuse the existing indexed documents to construct scheme-level representative profiles for Cohere Rerank.

In [5]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [7]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# print(f"Model loading time: {load_time:.2f} seconds")
# print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [9]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    persist_directory=str(DISCOVERY_DB_PATH),
    embedding_function=embedding_model
)

print("Discovery database loaded successfully.")
print("Collection count:", discovery_db._collection.count())

Discovery database loaded successfully.
Collection count: 23438


### Retrieve Discovery Documents

Fetch all indexed discovery chunks and their metadata from Chroma.

These documents will be grouped by scheme_id to construct representative scheme profiles.

In [13]:
from langchain_core.documents import Document

discovery_data = discovery_db.get(
    include=["documents", "metadatas"]
)

discovery_documents = [
    Document(
        page_content=content,
        metadata=metadata or {}
    )
    for content, metadata in zip(
        discovery_data["documents"],
        discovery_data["metadatas"]
    )
    if content and metadata and metadata.get("scheme_id")
]

print("Total indexed documents:", len(discovery_documents))
print(
    "Unique schemes:",
    len({
        doc.metadata["scheme_id"]
        for doc in discovery_documents
    })
)

Total indexed documents: 23438
Unique schemes: 4428


### Build Scheme-Level Representative Profiles

Create one representative document for each government scheme.

1. Group discovery chunks using scheme_id.
2. Remove duplicate chunks.
3. Combine available section content.
4. Preserve scheme ID and scheme name.
5. Limit representative text to 3,500 characters.

These profiles will be used by Cohere Rerank to compare user queries against individual schemes rather than individual chunks.

In [14]:
from collections import defaultdict

# Step 1: Group documents by scheme_id
scheme_chunks = defaultdict(list)
scheme_names = {}

for doc in discovery_documents:
    metadata = doc.metadata or {}

    scheme_id = metadata.get("scheme_id")
    scheme_name = metadata.get("scheme_name")

    if not scheme_id:
        continue

    scheme_chunks[scheme_id].append({
        "content": doc.page_content,
        "section": metadata.get("child_type", "unknown")
    })

    if scheme_name:
        scheme_names[scheme_id] = scheme_name


# Step 2: Build representative profiles
scheme_profiles = {}

MAX_PROFILE_CHARS = 3500

for scheme_id, chunks in scheme_chunks.items():

    seen = set()
    profile_sections = []

    scheme_name = scheme_names.get(scheme_id, "Unknown Scheme")

    for chunk in chunks:
        content = chunk["content"].strip()
        section = chunk["section"]

        if not content or content in seen:
            continue

        seen.add(content)

        profile_sections.append(
            f"Section: {section}\n{content}"
        )

    # Step 3: Combine sections
    representative_text = (
        f"Scheme: {scheme_name}\n"
        + "\n\n".join(profile_sections)
    )

    # Step 4: Limit profile size
    scheme_profiles[scheme_id] = representative_text[
        :MAX_PROFILE_CHARS
    ]


# Step 5: Validate profiles
print("Total scheme profiles:", len(scheme_profiles))

sample_scheme_id = next(iter(scheme_profiles))

print("\nSample scheme ID:", sample_scheme_id)
print("\nRepresentative text:")
print(scheme_profiles[sample_scheme_id])

Total scheme profiles: 4428

Sample scheme ID: LBezF5gB5uF80_MGGY81

Representative text:
Scheme: Unknown Scheme
Section: overview
Scheme: Stand-Up India
Section: overview
A scheme by Ministry of Finance for financing SC/ST and Women Entrepreneurs by facilitating bank loans for setting up a greenfield project Enterprise in manufacturing, services, trading sector and activities allied to agriculture.

Section: benefits
Scheme: Stand-Up India
Section: benefits
1. Facilitation of composite loan (inclusive of term loan and working capital) between ₹10 Lakhs and ₹100 Lakhs. Rupay debit card to be issued for convenience of the borrower.
1. The web portal by SIDBI provides hand-holding support through a network of agencies engaged in training, skill development, mentoring, project report preparation, application filling, work shed / utility support services, subsidy schemes etc.

Section: beneficiaries
Scheme: Stand-Up India
Section: beneficiaries
Individual; Business Entity

Section: eligibi

In [15]:
import time


# Project directories
PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

with open(GOLDEN_PATH, "r") as f:
    golden_df = pd.read_json(f, lines=True)

# Select one evaluation query
sample = hybrid_df.iloc[0]

question_lookup = golden_df.set_index("test_id")["question"].to_dict()


sample_query = question_lookup[sample["test_id"]]
candidate_ids = sample["ranked_scheme_ids"]

# Prepare scheme profiles
documents = []
valid_scheme_ids = []

for scheme_id in candidate_ids:
    profile = scheme_profiles.get(scheme_id)

    if profile:
        documents.append(profile)
        valid_scheme_ids.append(scheme_id)

print("Query:", sample_query)
print("Candidate schemes:", len(documents))

# Call Cohere Rerank API
start_time = time.perf_counter()

response = co.rerank(
    model=RERANKER_MODEL,
    query=sample_query,
    documents=documents,
    top_n=len(documents)
)

cohere_latency = time.perf_counter() - start_time

# Map API results to original scheme IDs
cohere_results = [
    {
        "scheme_id": valid_scheme_ids[result.index],
        "rerank_score": result.relevance_score,
        "rank": rank
    }
    for rank, result in enumerate(response.results, start=1)
]

print(f"\nAPI latency: {cohere_latency:.3f} seconds")
print("\nTop 5 Cohere results:")

for item in cohere_results[:5]:
    print(item)

Query: I need a loan to purchase a light motor vehicle for my transport business.
Candidate schemes: 37

API latency: 16.339 seconds

Top 5 Cohere results:
{'scheme_id': '3he_F5gB5uF80_MGg5Hz', 'rerank_score': 0.80486834, 'rank': 1}
{'scheme_id': 'hqIa0pwBXu-iUKXq00o3', 'rerank_score': 0.73671806, 'rank': 2}
{'scheme_id': 'msYJf5gB0AGc0F0ivjhP', 'rerank_score': 0.58376604, 'rank': 3}
{'scheme_id': 'ChdNbZoB5uF80_MGrpjI', 'rerank_score': 0.5578568, 'rank': 4}
{'scheme_id': 'RhhdDZwB5uF80_MG3Fwl', 'rerank_score': 0.5215216, 'rank': 5}


### Cohere Reranking with Rate-Limit Handling

Execute Cohere Rerank 5 Fast across the Hybrid RRF evaluation dataset.

- Limit requests to 6 per minute.
- Handle HTTP 429 using a 65-second cooldown.
- Retry temporary server errors.
- Save each successful query immediately.
- Resume automatically using test_id.
- Preserve API latency and retry information.

In [17]:
import json
import time
import pandas as pd

OUTPUT_PATH = (
    PROJECT_DIR
    / "notebooks/experiments/discovery_evaluation"
    / "cohere_rerank_results.jsonl"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

REQUEST_INTERVAL = 10
MAX_RETRIES = 8
MAX_QUERIES = None  # Change to None after testing

# Load completed checkpoints
completed_results = {}

if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                completed_results[str(record["test_id"])] = record
            except json.JSONDecodeError:
                continue

print("Previously completed:", len(completed_results))

pending_rows = [
    row for _, row in hybrid_df.iterrows()
    if str(row["test_id"]) not in completed_results
]

if MAX_QUERIES is not None:
    pending_rows = pending_rows[:MAX_QUERIES]

print("Pending queries:", len(pending_rows))

last_request_time = 0
total_start = time.perf_counter()

for batch_start in range(0, len(pending_rows), 10):

    batch = pending_rows[batch_start:batch_start + 10]

    print(f"\nProcessing batch {batch_start // 10 + 1}")

    for row in batch:

        test_id = str(row["test_id"])
        query = question_lookup[row["test_id"]]

        candidate_ids = [
            sid for sid in row["ranked_scheme_ids"]
            if sid in scheme_profiles
        ]

        documents = [
            scheme_profiles[sid]
            for sid in candidate_ids
        ]

        if not documents:
            continue

        retries = 0

        while True:

            # Respect request interval
            elapsed = time.monotonic() - last_request_time
            wait_time = REQUEST_INTERVAL - elapsed

            if wait_time > 0:
                time.sleep(wait_time)

            request_start = time.perf_counter()

            try:
                response = co.rerank(
                    model=RERANKER_MODEL,
                    query=query,
                    documents=documents,
                    top_n=len(documents)
                )

                api_latency = time.perf_counter() - request_start
                last_request_time = time.monotonic()
                break

            except Exception as exc:

                last_request_time = time.monotonic()

                status = getattr(exc, "status_code", None)

                if status is None:
                    status = getattr(
                        getattr(exc, "response", None),
                        "status_code",
                        None
                    )

                if status == 429:
                    retries += 1
                    wait = 65

                elif status in (500, 502, 503, 504):
                    retries += 1
                    wait = min(60, 5 * (2 ** (retries - 1)))

                else:
                    raise

                if retries > MAX_RETRIES:
                    raise

                print(
                    f"HTTP {status} | Retry {retries} | "
                    f"Waiting {wait}s"
                )

                time.sleep(wait)

        reranked_results = [
            {
                "scheme_id": candidate_ids[result.index],
                "rerank_score": float(result.relevance_score),
                "rank": rank
            }
            for rank, result in enumerate(
                response.results, start=1
            )
        ]

        record = {
            "test_id": test_id,
            "question": query,
            "relevant_scheme_ids": list(
                row["relevant_scheme_ids"]
            ),
            "candidate_scheme_ids": candidate_ids,
            "reranked_results": reranked_results,
            "reranked_scheme_ids": [
                item["scheme_id"]
                for item in reranked_results
            ],
            "api_latency_seconds": api_latency,
            "retry_count": retries,
            "model": RERANKER_MODEL
        }

        # Save immediately
        with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

        completed_results[test_id] = record

        print(
            f"Completed: {len(completed_results)} | "
            f"Latency: {api_latency:.2f}s | "
            f"Retries: {retries}"
        )

print("\nProcessing finished.")
print("Completed:", len(completed_results))
print("Saved:", OUTPUT_PATH)
print(
    "Total runtime:",
    round((time.perf_counter() - total_start) / 60, 2),
    "minutes"
)

Previously completed: 582
Pending queries: 23

Processing batch 1
Completed: 583 | Latency: 0.82s | Retries: 0
Completed: 584 | Latency: 0.73s | Retries: 0
Completed: 585 | Latency: 0.76s | Retries: 0
Completed: 586 | Latency: 0.94s | Retries: 0
Completed: 587 | Latency: 0.65s | Retries: 0
Completed: 588 | Latency: 0.70s | Retries: 0
Completed: 589 | Latency: 0.76s | Retries: 0
Completed: 590 | Latency: 0.64s | Retries: 0
Completed: 591 | Latency: 0.70s | Retries: 0
Completed: 592 | Latency: 0.59s | Retries: 0

Processing batch 2
Completed: 593 | Latency: 0.56s | Retries: 0
Completed: 594 | Latency: 0.57s | Retries: 0
Completed: 595 | Latency: 0.62s | Retries: 0
Completed: 596 | Latency: 1.52s | Retries: 0
Completed: 597 | Latency: 1.21s | Retries: 0
Completed: 598 | Latency: 2.53s | Retries: 0
Completed: 599 | Latency: 1.45s | Retries: 0
Completed: 600 | Latency: 1.10s | Retries: 0
Completed: 601 | Latency: 1.26s | Retries: 0
Completed: 602 | Latency: 1.08s | Retries: 0

Processing ba

### Cohere Reranking Results Validation

Verify that all 605 golden-dataset queries have completed reranking, that there are no duplicate test IDs, and that each query contains valid ranked scheme IDs.

In [21]:


COHERE_RESULTS_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\cohere_rerank_results.jsonl")

cohere_df = pd.read_json(COHERE_RESULTS_PATH, lines=True)

expected_ids = set(golden_df["test_id"])
completed_ids = set(cohere_df["test_id"])

print("Total records:", len(cohere_df))
print("Unique test IDs:", cohere_df["test_id"].nunique())
print("Missing queries:", len(expected_ids - completed_ids))
print("Unexpected queries:", len(completed_ids - expected_ids))
print("Empty reranked results:",
      cohere_df["reranked_scheme_ids"].apply(
          lambda x: not isinstance(x, list) or len(x) == 0
      ).sum())

print("\nAverage API latency:",
      round(cohere_df["api_latency_seconds"].mean(), 3), "seconds")

print("Total API retries:", cohere_df["retry_count"].sum())

display(
    cohere_df[
        [
            "test_id",
            "reranked_scheme_ids",
            "api_latency_seconds",
            "retry_count"
        ]
    ].head()
)

ValueError: Expected object or value at position 3200628

In [22]:
import json

valid_records = []
invalid_records = []

with open(COHERE_RESULTS_PATH, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            valid_records.append(json.loads(line))
        except json.JSONDecodeError as e:
            invalid_records.append({
                "line": line_number,
                "error": str(e),
                "content_preview": line[:300]
            })

print("Valid records:", len(valid_records))
print("Invalid records:", len(invalid_records))

for item in invalid_records[:5]:
    print("\n", item)

Valid records: 605
Invalid records: 1

 {'line': 580, 'error': 'Expecting value: line 1 column 1 (char 0)', 'content_preview': '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00

In [23]:
cohere_df = pd.DataFrame(valid_records)

print("Records:", len(cohere_df))
print("Unique test IDs:", cohere_df["test_id"].nunique())

Records: 605
Unique test IDs: 605


### Recover Cohere Reranking Results

Create a clean JSONL checkpoint from all valid records. Preserve the original file for debugging and recovery.

In [26]:
import shutil
import json

RESULTS_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation")

BACKUP_PATH = RESULTS_DIR / "cohere_rerank_results_backup.jsonl"
CLEAN_PATH = RESULTS_DIR / "cohere_rerank_results_clean.jsonl"

# Preserve original checkpoint
shutil.copy2(COHERE_RESULTS_PATH, BACKUP_PATH)

# Write validated records
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    for record in valid_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Verify recovered file
with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    recovered_records = [json.loads(line) for line in f if line.strip()]

recovered_ids = {r["test_id"] for r in recovered_records}

print("Recovered records:", len(recovered_records))
print("Unique IDs:", len(recovered_ids))
# print("Missing IDs:", len(expected_ids - recovered_ids))
print("Backup:", BACKUP_PATH)
print("Clean file:", CLEAN_PATH)

Recovered records: 605
Unique IDs: 605
Backup: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\cohere_rerank_results_backup.jsonl
Clean file: G:\SEVA_Ai\SEVA_Ai\notebooks\experiments\discovery_evaluation\cohere_rerank_results_clean.jsonl


In [27]:
expected_ids = set(golden_df["test_id"])
recovered_ids = {r["test_id"] for r in recovered_records}

print("Expected queries:", len(expected_ids))
print("Recovered queries:", len(recovered_ids))
print("Missing queries:", len(expected_ids - recovered_ids))
print("Unexpected queries:", len(recovered_ids - expected_ids))

assert expected_ids == recovered_ids, "Query ID mismatch!"

cohere_df = pd.DataFrame(recovered_records)

print("\nValidation successful!")
print("DataFrame shape:", cohere_df.shape)

Expected queries: 605
Recovered queries: 605
Missing queries: 0
Unexpected queries: 0

Validation successful!
DataFrame shape: (605, 10)


### Normalize Cohere Reranking Results

Unify latency measurements from different checkpoint versions while preserving missing retry and model information. Validate the ranking and latency fields before calculating retrieval metrics.

In [31]:
# Combine latency fields without losing recorded values
cohere_df["rerank_latency_seconds"] = (
    cohere_df["api_latency_seconds"]
    .combine_first(cohere_df["latency"])
)

# Validate normalized results
print("Total queries:", len(cohere_df))
print("Unique queries:", cohere_df["test_id"].nunique())

print(
    "Missing normalized latency:",
    cohere_df["rerank_latency_seconds"].isna().sum()
)

print(
    "Average reranking latency:",
    round(cohere_df["rerank_latency_seconds"].mean(), 3),
    "seconds"
)

print(
    "P50 latency:",
    round(cohere_df["rerank_latency_seconds"].quantile(0.50), 3)
)

print(
    "P95 latency:",
    round(cohere_df["rerank_latency_seconds"].quantile(0.95), 3)
)

print(
    "P99 latency:",
    round(cohere_df["rerank_latency_seconds"].quantile(0.99), 3)
)

print(
    "Retry counts available:",
    cohere_df["retry_count"].notna().sum()
)

print("\nModel distribution:")
print(cohere_df["model"].value_counts(dropna=False))

Total queries: 605
Unique queries: 605
Missing normalized latency: 0
Average reranking latency: 1.449 seconds
P50 latency: 0.742
P95 latency: 3.912
P99 latency: 18.758
Retry counts available: 595

Model distribution:
model
rerank-v5.0-fast    595
NaN                  10
Name: count, dtype: int64


### Cohere Reranking Evaluation

Evaluate the ranking quality of Cohere reranked results using Ragas ID-based Context Precision and Recall, along with MRR, MAP, and NDCG.

Evaluate at K = 5, 10, 20, and 30.

In [33]:
from ragas import SingleTurnSample
import numpy as np
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall

id_precision = IDBasedContextPrecision()
id_recall = IDBasedContextRecall()

TOP_K_VALUES = [5, 10, 20, 30]

def calculate_ranking_metrics(ranked_ids, relevant_ids, k):
    ranked = ranked_ids[:k]
    relevant = set(relevant_ids)

    if not relevant:
        return {}

    hits = [1 if item in relevant else 0 for item in ranked]

    # Precision@K and Recall@K
    precision = sum(hits) / k
    recall = sum(hits) / len(relevant)

    # Reciprocal Rank
    reciprocal_rank = 0.0
    for rank, hit in enumerate(hits, start=1):
        if hit:
            reciprocal_rank = 1 / rank
            break

    # Average Precision
    relevant_hits = 0
    precision_sum = 0.0

    for rank, hit in enumerate(hits, start=1):
        if hit:
            relevant_hits += 1
            precision_sum += relevant_hits / rank

    ap = precision_sum / min(len(relevant), k)

    # NDCG
    dcg = sum(
        hit / np.log2(rank + 1)
        for rank, hit in enumerate(hits, start=1)
    )

    ideal_hits = min(len(relevant), k)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    ndcg = dcg / idcg if idcg else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "mrr": reciprocal_rank,
        "map": ap,
        "ndcg": ndcg
    }


# Check relevance-label distribution
relevance_counts = cohere_df["relevant_scheme_ids"].apply(len)

print("Relevant schemes per query:")
print(relevance_counts.value_counts().sort_index())


# Calculate metrics
evaluation_results = []

for k in TOP_K_VALUES:
    query_metrics = []
    ragas_precision_scores = []
    ragas_recall_scores = []

    for record in cohere_df.to_dict("records"):
        ranked_ids = record["reranked_scheme_ids"]
        relevant_ids = record["relevant_scheme_ids"]

        metrics = calculate_ranking_metrics(
            ranked_ids,
            relevant_ids,
            k
        )

        query_metrics.append(metrics)

        sample = SingleTurnSample(
            retrieved_context_ids=ranked_ids[:k],
            reference_context_ids=relevant_ids
        )

        ragas_p = await id_precision.single_turn_ascore(sample)
        ragas_r = await id_recall.single_turn_ascore(sample)

        ragas_precision_scores.append(float(ragas_p))
        ragas_recall_scores.append(float(ragas_r))

    evaluation_results.append({
        "retrieval_method": "Hybrid_RRF_Cohere",
        "k": k,
        "ragas_precision": np.mean(ragas_precision_scores),
        "ragas_recall": np.mean(ragas_recall_scores),
        "mrr": np.mean([m["mrr"] for m in query_metrics]),
        "map": np.mean([m["map"] for m in query_metrics]),
        "ndcg": np.mean([m["ndcg"] for m in query_metrics])
    })

cohere_metrics_df = pd.DataFrame(evaluation_results)

display(
    cohere_metrics_df.round(4)
)

cohere_metrics_df.to_csv(
    RESULTS_DIR / "cohere_rerank_evaluation.csv",
    index=False
)

C:\Users\devendra\AppData\Local\Temp\ipykernel_7656\86528707.py:3: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
C:\Users\devendra\AppData\Local\Temp\ipykernel_7656\86528707.py:3: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall


Relevant schemes per query:
relevant_scheme_ids
1    605
Name: count, dtype: int64


,retrieval_method,k,ragas_precision,ragas_recall,mrr,map,ndcg
0,Hybrid_RRF_Cohere,5,0.1276,0.6380,0.4972,0.4972,0.5326
1,Hybrid_RRF_Cohere,10,0.0676,0.6760,0.5025,0.5025,0.5452
2,Hybrid_RRF_Cohere,20,0.0348,0.6942,0.5038,0.5038,0.5498
3,Hybrid_RRF_Cohere,30,0.0241,0.6959,0.5039,0.5039,0.5502


### Unified Retrieval Comparison

Combine Ragas ID-based Context Precision and Context Recall results for Dense, BM25, Hybrid RRF, and Hybrid RRF with Cohere reranking.

In [35]:
# Normalize existing results
baseline_metrics = comparison_df.rename(
    columns={
        "retrieval": "retrieval_method",
        "K": "k",
        "context_precision": "precision",
        "context_recall": "recall"
    }
)[["retrieval_method", "k", "precision", "recall"]].copy()

# Normalize Cohere results
cohere_ragas = cohere_metrics_df.rename(
    columns={
        "ragas_precision": "precision",
        "ragas_recall": "recall"
    }
)[["retrieval_method", "k", "precision", "recall"]].copy()

# Combine all methods
all_ragas_metrics = pd.concat(
    [baseline_metrics, cohere_ragas],
    ignore_index=True
)

all_ragas_metrics = all_ragas_metrics.sort_values(
    ["k", "retrieval_method"]
).reset_index(drop=True)

display(all_ragas_metrics.round(4))

all_ragas_metrics.to_csv(
    RESULTS_DIR / "all_retrieval_ragas_comparison.csv",
    index=False
)

print("Total evaluation rows:", len(all_ragas_metrics))

,retrieval_method,k,precision,recall
0,BM25,5,0.0833,0.4165
1,Dense,5,0.0618,0.3091
2,Hybrid RRF,5,0.0826,0.4132
3,Hybrid_RRF_Cohere,5,0.1276,0.6380
4,BM25,10,0.0491,0.4909
5,Dense,10,0.0397,0.3967
6,Hybrid RRF,10,0.0512,0.5124
7,Hybrid_RRF_Cohere,10,0.0676,0.6760
8,BM25,20,0.0292,0.5603
9,Dense,20,0.0263,0.4694


Total evaluation rows: 16
